# 03 — Baseline y cinco candidatos
Primera comparación sin tuning: **Ridge, Random Forest, XGBoost, MLP y SVR**, más `DummyRegressor` que predice la media del PKD de entrenamiento como referencia mínima.

Usamos los mismos cinco folds por grupos. La imputación y el escalado se ajustan desde cero dentro de cada fold. Los 293 casos sin representación solo participan en entrenamiento, nunca en los scores de validación. Las métricas describen por tanto el subconjunto con grupos conocidos.

**RMSE** es el criterio técnico provisional de comparación (menor es mejor); también registramos **MAE** y **R²**. Mostramos media y desviación entre folds, no un intervalo de confianza. No modificamos el criterio de negocio pendiente del PSet1. No usamos test, no optimizamos hiperparámetros y no elegimos aún un modelo final.

**Corrección del protocolo:** estimadores con hiperparámetros por defecto de las versiones instaladas; solo se fija semilla y paralelismo. Los pipelines conservan el escalado necesario para Ridge, MLP y SVR; esta diferencia de preparación está declarada, no es tuning. Las corridas anteriores permanecen en MLflow como exploración; la comparación vigente usa el tag `protocol=defaults_v2`.


In [1]:
from pathlib import Path
import sys, json, hashlib
import pandas as pd
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
from src.pipeline import construir_pipeline
from src.splitting import folds_por_grupos
config = json.loads((ROOT / "configs/pipelines.json").read_text())
snapshot = Path(json.loads((ROOT / "configs/dataset.local.json").read_text())["snapshot"]).name
carpeta = ROOT / "data/modelado" / snapshot
manifest = json.loads((carpeta / "manifest.json").read_text())
split_manifest = json.loads((carpeta / "split/manifest.json").read_text())
for archivo, esperado in [(carpeta / "datos_modelado.parquet", split_manifest["source_sha256"]),
                          (carpeta / "split/asignacion.parquet", split_manifest["assignment_sha256"])]:
    assert hashlib.sha256(archivo.read_bytes()).hexdigest() == esperado, f"Archivo modificado: {archivo.name}"
datos = pd.read_parquet(carpeta / "datos_modelado.parquet")
asignacion = pd.read_parquet(carpeta / "split/asignacion.parquet")
assert datos.index.equals(asignacion.index)
features = manifest["predictors"]
train = datos.loc[asignacion["PARTICION"].eq("train")].copy()
X_train = train[features]
y_train = train[manifest["target"]]
grupos_train = train[manifest["group"]]
solo_train = asignacion.loc[train.index, "ENTRENAMIENTO_OBLIGATORIO"]
print("Entrenamiento:", X_train.shape, "| Test reservado, sin analizar")


Entrenamiento: (3670, 173) | Test reservado, sin analizar


## 1. Validación reproducible
Los folds se construyen con el mismo orden de filas y la misma regla del cuaderno 02. Guardamos además las asignaciones y sus hashes para auditar cada ejecución.


In [2]:
from src.experimentos import comparar_candidatos
folds = folds_por_grupos(grupos_train, solo_train, n_splits=config["cv_folds"])
CARPETA_RESULTADOS = ROOT / "reports" / "baseline" / snapshot
CARPETA_RESULTADOS.mkdir(parents=True, exist_ok=True)
asignaciones_cv = pd.concat([
    pd.DataFrame({"fila_modelo": X_train.index[va], "fold_validacion": i+1})
    for i, (_, va) in enumerate(folds)
], ignore_index=True)
assert asignaciones_cv.fila_modelo.is_unique
assert len(asignaciones_cv) == int((~solo_train).sum())
asignaciones_cv.to_csv(CARPETA_RESULTADOS / "folds.csv", index=False)
import subprocess
import sklearn, xgboost, numpy
try:
    git_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True, stderr=subprocess.DEVNULL).strip()
except (OSError, subprocess.CalledProcessError):
    git_commit = "unavailable"
contexto = {
    "snapshot":snapshot, "protocol":"defaults_v2", "git_commit":git_commit,
    "versions":{"sklearn":sklearn.__version__, "xgboost":xgboost.__version__, "numpy":numpy.__version__},
    "experiments_code_sha256":hashlib.sha256((ROOT / "src/experimentos.py").read_bytes()).hexdigest(), "source_sha256":split_manifest["source_sha256"],
    "split_assignment_sha256":split_manifest["assignment_sha256"],
    "folds_sha256":hashlib.sha256((CARPETA_RESULTADOS / "folds.csv").read_bytes()).hexdigest(),
    "pipeline_code_sha256":hashlib.sha256((ROOT / "src/pipeline.py").read_bytes()).hexdigest(),
    "test_used":False, "mandatory_train_rows":int(solo_train.sum()),
    "primary_metric_provisional":"RMSE", "transformations":"none",
}
print("Filas de entrenamiento:",len(X_train),"| Filas evaluables en CV:",len(asignaciones_cv))


Filas de entrenamiento: 3670 | Filas evaluables en CV: 3377


## 2. Entrenar candidatos y registrar en MLflow
Esta celda entrena 30 estimadores (seis candidatos × cinco folds). Reejecutarla crea nuevos runs de MLflow y actualiza los CSV locales. No se guarda un modelo para deployment en esta fase.


In [3]:
resumen_candidatos, metricas_folds = comparar_candidatos(
    X_train, y_train, folds, config["modelos"], CARPETA_RESULTADOS, contexto, seed=config["seed"],
)
columnas_comparacion = ["train_rmse_mean", "rmse_mean", "rmse_std", "gap_rmse_mean",
                        "train_mae_mean", "mae_mean", "train_r2_mean", "r2_mean", "fit_seconds_total"]
display(resumen_candidatos[columnas_comparacion].round(4))
candidatos_tuning = resumen_candidatos.drop(index="DummyMean").head(3).index.tolist()
print("Tres candidatos propuestos por RMSE de CV:", candidatos_tuning)


Evaluando DummyMean...


  fold 1: RMSE=1.680


  fold 2: RMSE=1.971


  fold 3: RMSE=1.831


  fold 4: RMSE=2.149


  fold 5: RMSE=1.811


... (40 outputs left out)

2026/11/19 02:31:53 INFO mlflow.tracking._tracking_service.client: 🏃 View run baseline_SVR at: http://mlflow:5000/#/experiments/1/runs/5f54e17b53e141cfa4a86ffd8729637f.


2026/11/19 02:31:54 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://mlflow:5000/#/experiments/1.


,train_rmse_mean,rmse_mean,rmse_std,gap_rmse_mean,train_mae_mean,mae_mean,train_r2_mean,r2_mean,fit_seconds_total
modelo,,,,,,,,,
RandomForestRegressor,0.4862,1.6325,0.1136,1.1463,0.3493,1.2581,0.9344,0.2371,20.8876
SVR,1.2204,1.6503,0.0950,0.4299,0.7879,1.2565,0.5865,0.2176,18.3464
XGBRegressor,0.0965,1.7067,0.0925,1.6102,0.0712,1.3180,0.9974,0.1610,8.0281
Ridge,1.6456,1.7980,0.1022,0.1524,1.2927,1.4148,0.2488,0.0704,0.4621
DummyMean,1.8991,1.8884,0.1785,-0.0106,1.5012,1.5037,0.0000,-0.0151,0.0063
MLPRegressor,0.3396,2.0132,0.1188,1.6736,0.2345,1.5800,0.9677,-0.1690,62.3594


Tres candidatos propuestos por RMSE de CV: ['RandomForestRegressor', 'SVR', 'XGBRegressor']


## 3. Lectura de resultados
Comparar RMSE y MAE contra DummyMean; un R² negativo significa rendimiento peor que predecir la media de las etiquetas de ese fold de validación según la definición de R². Revisar también dispersión entre folds y advertencias de convergencia, especialmente en MLP.

Las predicciones fuera de fold y los scores quedan en `reports/baseline/<snapshot>/` y en MLflow. La clasificación obtenida es exploratoria para orientar tuning con entrenamiento; no es un resultado de test ni una selección definitiva. La similitud no medida de los 293 casos obligatorios sigue siendo una limitación.

**Brechas:** `gap_rmse = RMSE_CV − RMSE_train`, `gap_mae = MAE_CV − MAE_train` y `gap_r2 = R²_train − R²_CV`; una brecha positiva grande puede señalar sobreajuste. Son promedios entre los mismos cinco folds. Las métricas train incluyen los casos obligatorios; CV solo contiene grupos conocidos, por lo que la brecha también refleja esa diferencia de población. `fit_seconds_total` suma únicamente los cinco ajustes, sin predicción ni registro en MLflow. Las advertencias se conservan; no se aumenta `max_iter` para ocultarlas en esta fase por defecto.


In [4]:
advertencias = json.loads((CARPETA_RESULTADOS / "advertencias.json").read_text())
if advertencias:
    display(pd.DataFrame(advertencias).drop_duplicates(["modelo", "categoria", "mensaje"]))
else:
    print("Sin advertencias durante el entrenamiento.")


,modelo,fold,categoria,mensaje
0,MLPRegressor,1,ConvergenceWarning,Stochastic Optimizer: Maximum iterations (200)...
